# Train the Open Breast Ultrasound baseline in Colab

This notebook is a research workflow for the open-breast-ultrasound package. It is not a medical device, and it does not report a diagnosis.

A Colab session can be interrupted or recycled. Files written only to the Colab disk are not durable. Copy the experiment directory to Google Drive, or download a zip, before the session ends.

The notebook never sets `reviewed=true` for you. `reviewed=true` means a person looked at an inventory row. It is not a medical validation.


## 1. Check whether a GPU is visible

CPU training is possible and slow. A missing GPU is not an error.


In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow", tf.__version__)
print("GPUs:", gpus if gpus else "none detected")


## 2. Install the project

Use one of the two options in the next cell. Do not do both.

- Option A: upload `open-breast-ultrasound.zip` if the repository is not public yet.
- Option B: set `REPOSITORY_URL` to a git URL you control, then clone it.

The archive or clone must contain `pyproject.toml`.


In [ ]:
from pathlib import Path
import os

REPOSITORY_URL = ""  # example: https://github.com/YOUR_ACCOUNT/open-breast-ultrasound.git
ROOT = Path("/content/open-breast-ultrasound")

if REPOSITORY_URL.strip():
    if ROOT.exists():
        print(f"Using existing directory {ROOT}")
    else:
        !git clone "{REPOSITORY_URL}" "{ROOT}"
elif not ROOT.exists():
    from google.colab import files
    uploaded = files.upload()
    zips = [name for name in uploaded if name.endswith(".zip")]
    if len(zips) != 1:
        raise RuntimeError("Upload exactly one .zip of the repository, or set REPOSITORY_URL.")
    zip_path = Path("/content") / zips[0]
    zip_path.write_bytes(uploaded[zips[0]])
    !unzip -q -o "{zip_path}" -d /content
    del uploaded
    if not ROOT.exists():
        candidates = list(Path("/content").glob("*/pyproject.toml"))
        if len(candidates) != 1:
            raise RuntimeError("Could not find the extracted project. Expected open-breast-ultrasound/pyproject.toml.")
        ROOT = candidates[0].parent

os.chdir(ROOT)
print("Project root:", Path.cwd())
!python -m pip install -q -e ".[dev]"


## 3. Choose the dataset folder

Either download BUSI from Kaggle or set `DATA_ROOT` to a folder that is already on Drive or in the session.

Kaggle authentication reads `kaggle.json` from the upload picker or from `~/.kaggle/kaggle.json`. The notebook does not print the file. Do not paste the token into a cell.

The dataset terms are not the Apache-2.0 license of this code. Accept them on Kaggle before downloading. Cite Al-Dhabyani et al., Data in Brief 2020;28:104863.


In [ ]:
from pathlib import Path
import os
import stat

DATA_ROOT = ""  # set this to an existing folder to skip the Kaggle download
DOWNLOAD_FROM_KAGGLE = not bool(DATA_ROOT.strip())

if DOWNLOAD_FROM_KAGGLE:
    kaggle_path = Path.home() / ".kaggle" / "kaggle.json"
    if not kaggle_path.is_file():
        from google.colab import files
        uploaded = files.upload()
        credential_names = [name for name in uploaded if name.endswith(".json")]
        if len(credential_names) != 1:
            raise RuntimeError("Upload kaggle.json, or set DATA_ROOT and skip this download.")
        kaggle_path.parent.mkdir(parents=True, exist_ok=True)
        kaggle_path.write_bytes(uploaded[credential_names[0]])
        kaggle_path.chmod(stat.S_IRUSR | stat.S_IWUSR)
        del uploaded
        print("Stored Kaggle credentials without printing them.")
    download_dir = Path("/content/busi-download")
    download_dir.mkdir(parents=True, exist_ok=True)
    !kaggle datasets download -d aryashah2k/breast-ultrasound-images-dataset -p "{download_dir}"
    zip_files = list(download_dir.glob("*.zip"))
    if len(zip_files) != 1:
        raise RuntimeError(f"Expected one Kaggle zip, found {zip_files}")
    !unzip -q -o "{zip_files[0]}" -d "{download_dir}"
    DATA_ROOT = str(download_dir)

print("DATA_ROOT =", DATA_ROOT)
import os
os.environ["DATA_ROOT"] = str(DATA_ROOT)


## 4. Audit

This does not delete images or masks. It does not mark any row as reviewed. Filename numbers are not patient IDs.


In [ ]:
!python -m open_breast_ultrasound.cli audit \
  --data-root "$DATA_ROOT" \
  --output-dir outputs/audit


## 5. Stop and review the inventory

Open `outputs/audit/inventory.csv` and `outputs/duplicate_candidates.csv`.

- Set `reviewed` to `true` only for rows you have actually looked at.
- Set `include` to `false` for rows you want to drop, and write the reason in `review_note`.
- Do not change a label unless you have a documented reason. This notebook will not change labels for you.
- Pawłowska, Karwat, and Żołek, Data in Brief 2023;48:109247, report copies, non-breast images, and questionable labels. Applying that list is a human step.

`reviewed=true` is not a medical validation.

The next cell is a real pause. It raises an error until you edit the flag yourself after the CSV edit. Leaving the flag false is the safe default.


In [ ]:
# Edit the CSV first. Then change this flag yourself.
# Do not replace this cell with code that sets reviewed=true on every row.
INVENTORY_REVIEWED_BY_ME = False

if not INVENTORY_REVIEWED_BY_ME:
    raise RuntimeError(
        "Stop. Review outputs/audit/inventory.csv, set the reviewed column yourself, "
        "then set INVENTORY_REVIEWED_BY_ME = True and run this cell again. "
        "reviewed=true is not a medical validation, and this notebook will not mark the dataset for you."
    )

!python -m open_breast_ultrasound.cli check-review --inventory outputs/audit/inventory.csv


## 6. Split

The seed is an input, not a value searched against a test score. Linked ids and exact duplicates stay together. This is not patient-level independence unless you supplied real patient IDs, and even then the software does not verify them.


In [ ]:
!python -m open_breast_ultrasound.cli split \
  --inventory outputs/audit/inventory.csv \
  --output-dir outputs/splits \
  --seed 42


## 7. Train

Test images are held out of fitting, early stopping, and model selection. The command refuses to overwrite an existing experiment directory.


In [ ]:
!python -m open_breast_ultrasound.cli train \
  --config configs/baseline.yaml \
  --data-root "$DATA_ROOT" \
  --inventory outputs/audit/inventory.csv \
  --split outputs/splits/split_manifest.csv \
  --output-dir outputs/experiments/baseline-seed42 \
  --confirm-review


## 8. Calibrate on validation only, then evaluate the test split separately

Calibration does not read test labels to fit the temperature. Evaluation does not refit the model, the temperature, or the threshold.


In [ ]:
!python -m open_breast_ultrasound.cli calibrate \
  --data-root "$DATA_ROOT" \
  --experiment outputs/experiments/baseline-seed42 \
  --inventory outputs/audit/inventory.csv \
  --split outputs/splits/split_manifest.csv

!python -m open_breast_ultrasound.cli evaluate \
  --data-root "$DATA_ROOT" \
  --experiment outputs/experiments/baseline-seed42 \
  --inventory outputs/audit/inventory.csv \
  --split outputs/splits/split_manifest.csv \
  --bootstrap 1000


## 9. Keep the outputs

Colab will delete the local disk when the runtime stops. Mount Drive and copy the experiment, or download a zip. Do not download the raw BUSI images as if this project had the right to redistribute them; copy your own experiment outputs.


In [ ]:
from pathlib import Path
import shutil

EXPERIMENT = Path("outputs/experiments/baseline-seed42")
archive = Path("/content/open-breast-ultrasound-experiment.zip")
shutil.make_archive(archive.with_suffix(""), "zip", EXPERIMENT)
print("Wrote", archive, "size", archive.stat().st_size)

SAVE_TO_DRIVE = False  # set True if you want a Drive copy
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    destination = Path("/content/drive/MyDrive/open-breast-ultrasound-baseline-seed42")
    if destination.exists():
        raise RuntimeError(f"Refusing to overwrite {destination}")
    shutil.copytree(EXPERIMENT, destination)
    print("Copied to", destination)
else:
    from google.colab import files
    files.download(str(archive))


## 10. Optional local demo

Public sharing stays off. Set `ALLOW_PUBLIC_SHARE` only if every image is non-identifying research material and you accept a third-party tunnel. Do not use it for clinical uploads.

The demo shows model scores. It does not diagnose, localize, or claim an early-detection benefit.


In [ ]:
from pathlib import Path
from open_breast_ultrasound.demo import launch_demo

ALLOW_PUBLIC_SHARE = False
launch_demo(
    Path("outputs/experiments/baseline-seed42"),
    host="0.0.0.0",
    port=7860,
    allow_public_share=ALLOW_PUBLIC_SHARE,
)
